## General Evaluation of Chatbot


In [ ]:
from langsmith import Client

ls_client = Client()

dataset_name = "JUSNL Tariff Petition Evaluation Dataset"

dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Evaluation dataset for Regulatory RAG on JUSNL Tariff Petition"
)

ls_client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {
                "question": "What is JUSNL?"
            },
            "outputs": {
                "answer": "Jharkhand Urja Sancharan Nigam Limited (JUSNL) is the State Transmission Utility responsible for power transmission in Jharkhand."
            }
        },

        {
            "inputs": {
                "question": "Why has JUSNL filed this petition?"
            },
            "outputs": {
                "answer": "JUSNL filed this petition for provisional true-up of FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26, and tariff determination."
            }
        },

        {
            "inputs": {
                "question": "What regulations has JUSNL relied upon?"
            },
            "outputs": {
                "answer": "JUSNL relied upon the Electricity Act 2003 and JSERC Transmission Tariff Regulations 2020."
            }
        },

        {
            "inputs": {
                "question": "Why is JUSNL filing provisional true-up for FY 2023-24?"
            },
            "outputs": {
                "answer": "The statutory audit for FY 2023-24 was not completed, therefore JUSNL filed a provisional true-up based on unaudited accounts."
            }
        },

        {
            "inputs": {
                "question": "What petitions has JUSNL filed?"
            },
            "outputs": {
                "answer": "JUSNL filed Provisional True-Up for FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26 and Tariff Proposal for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What employee expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected total employee expenses of Rs. 107.54 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What A&G expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected A&G expenses of Rs. 25.45 Crore."
            }
        },

        {
            "inputs": {
                "question": "What depreciation expenses has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed depreciation expenses based on asset capitalization and applicable depreciation rates under JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What return on equity has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Return on Equity as per JSERC Transmission Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "What ARR has JUSNL projected for FY 2025-26?"
            },
            "outputs": {
                "answer": "JUSNL projected a Gross ARR of Rs. 475.40 Crore and a total revenue requirement of Rs. 1,418.88 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What transmission charges has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed monthly transmission charges to be recovered from beneficiaries based on the approved ARR."
            }
        },

        {
            "inputs": {
                "question": "What capital expenditure schemes has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed various transmission system strengthening and network expansion schemes."
            }
        },

        {
            "inputs": {
                "question": "What O&M expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected Operation and Maintenance expenses comprising employee cost, A&G expenses and R&M expenses."
            }
        },

        {
            "inputs": {
                "question": "What Repair and Maintenance expenses has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Repair and Maintenance expenses as part of O&M expenses under JSERC norms."
            }
        },

        {
            "inputs": {
                "question": "What interest on loan has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected interest on loan based on outstanding debt and applicable interest rates."
            }
        },

        {
            "inputs": {
                "question": "What interest on working capital has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Interest on Working Capital according to JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What is the regulatory framework applicable to JUSNL?"
            },
            "outputs": {
                "answer": "The regulatory framework consists of the Electricity Act 2003 and JSERC Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "How does JUSNL calculate employee expenses?"
            },
            "outputs": {
                "answer": "Employee expenses are projected using inflation escalation factors and estimated manpower additions."
            }
        },

        {
            "inputs": {
                "question": "How are terminal benefits treated in employee expenses?"
            },
            "outputs": {
                "answer": "Terminal benefits are separately accounted for and included in total employee expenses."
            }
        },

        {
            "inputs": {
                "question": "What are the major components of ARR?"
            },
            "outputs": {
                "answer": "ARR consists of O&M expenses, depreciation, interest on loan, return on equity and other approved costs."
            }
        }
    ]
)

## Correctness

In [ ]:
from google import genai
import os

gemini_client  = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

eval_instructions = """
You are an expert professor specialized in grading students' answers to questions.
"""

def correctness(inputs, outputs, reference_outputs):

    prompt = f"""
You are grading the following question:

Question:
{inputs['question']}

Ground Truth Answer:
{reference_outputs['answer']}

Predicted Answer:
{outputs['response']}

Respond ONLY with:
CORRECT
or
INCORRECT
"""

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            eval_instructions,
            prompt
        ]
    )

    grade = response.text.strip().upper()

    return grade == "CORRECT"

## Concision

In [ ]:
def concision(outputs: dict, reference_outputs: dict) -> bool:
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))

## Evaluation Run

In [ ]:
from google import genai
import os


default_instructions = (
    "Respond to the user's question in a short, concise manner "
    "(one short sentence)."
)

def my_app(
    question: str,
    model: str = "gemini-2.5-flash",
    instructions: str = default_instructions
) -> str:

    prompt = f"""
{instructions}

Question:
{question}
"""

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt
    )

    return response.text.strip()

In [ ]:
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [ ]:
from langsmith import Client
from langsmith.evaluation import evaluate

client = Client()

results = evaluate(
    my_app,
    data=dataset_name,
    evaluators=[correctness],
    experiment_prefix="gemini-rag"
)

In [ ]:
"""
LangSmith Chatbot Evaluator — Gemini Edition (google-genai SDK)
===============================================================
Install:
    pip install -U langsmith google-genai

Env vars:
    LANGSMITH_TRACING=true
    LANGSMITH_API_KEY=<your key>
    GOOGLE_API_KEY=<your key>

On second+ runs: comment out the create_dataset / create_examples block.
"""

import os
from google import genai
from google.genai import types
from langsmith import Client

# ─────────────────────────────────────────────
# 1. Configure Gemini (new google-genai SDK)
# ─────────────────────────────────────────────
client_gemini = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


def call_gemini(
    prompt: str,
    model: str = "gemini-2.0-flash",
    system: str = "",
) -> str:
    """Call Gemini and return plain text."""
    config = types.GenerateContentConfig(
        temperature=0,
        system_instruction=system if system else None,
    )
    response = client_gemini.models.generate_content(
        model=model,
        contents=prompt,
        config=config,
    )
    return response.text.strip()


# ─────────────────────────────────────────────
# 2. LangSmith client + dataset
# ─────────────────────────────────────────────
client_ls = Client()
DATASET_NAME = "QA Example Dataset (Gemini)"

# ── Comment out after first run ──
dataset = client_ls.create_dataset(DATASET_NAME)
client_ls.create_examples(
    dataset_id=dataset.id,
    examples=[
        {"inputs": {"question": "What is LangChain?"},
         "outputs": {"answer": "A framework for building LLM applications"}},
        {"inputs": {"question": "What is LangSmith?"},
         "outputs": {"answer": "A platform for observing and evaluating LLM applications"}},
        {"inputs": {"question": "What is Google?"},
         "outputs": {"answer": "A technology company known for search"}},
        {"inputs": {"question": "What is Mistral?"},
         "outputs": {"answer": "A company that creates Large Language Models"}},
        {"inputs": {"question": "What is Gemini?"},
         "outputs": {"answer": "A family of large language models created by Google DeepMind"}},
    ],
)
print(f"✅ Dataset '{DATASET_NAME}' created with 5 examples.")
# ── End of first-run-only block ──


# ─────────────────────────────────────────────
# 3. Evaluators
# ─────────────────────────────────────────────
EVAL_SYSTEM = "You are an expert professor specialized in grading students' answers to questions."


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """LLM-as-a-judge using Gemini 2.0 Flash."""
    prompt = f"""You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with CORRECT or INCORRECT:
Grade:"""
    verdict = call_gemini(prompt, system=EVAL_SYSTEM)
    return verdict.strip().upper() == "CORRECT"


def concision(outputs: dict, reference_outputs: dict) -> bool:
    """Response must be shorter than 2× the reference answer length."""
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))


# ─────────────────────────────────────────────
# 4. App targets
# ─────────────────────────────────────────────
DEFAULT_INSTRUCTIONS = (
    "Respond to the user's question in a short, concise manner (one short sentence)."
)
STRICT_INSTRUCTIONS = (
    "Respond to the user's question in a short, concise manner (one short sentence). "
    "Do NOT use more than ten words."
)


def my_app(question: str, model: str = "gemini-2.0-flash", instructions: str = DEFAULT_INSTRUCTIONS) -> str:
    return call_gemini(question, model=model, system=instructions)


def target_flash(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="gemini-2.0-flash")}


# AFTER
def target_pro(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="gemini-2.0-flash")}

def target_pro_strict(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="gemini-2.0-flash", instructions=STRICT_INSTRUCTIONS)}

# ─────────────────────────────────────────────
# 5. Run evaluations
# ─────────────────────────────────────────────
print("\n🚀 Experiment 1: gemini-2.0-flash (default prompt) …")
r1 = client_ls.evaluate(
    target_flash,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="gemini-flash",
)
print(f"   → {r1.experiment_name}")

print("\n🚀 Experiment 2: gemini-2.5-pro (default prompt) …")
r2 = client_ls.evaluate(
    target_pro,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="gemini-flash-default",   # changed
)

print(f"   → {r2.experiment_name}")

print("\n🚀 Experiment 3: gemini-2.5-pro (strict prompt) …")
r3 = client_ls.evaluate(
    target_pro_strict,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="gemini-flash-strict",    # changed
)
print(f"   → {r3.experiment_name}")

print("\n✅ Done. Open LangSmith → Datasets & Testing to compare results.")

In [ ]:
"""
LangSmith Chatbot Evaluator — Groq Edition
==========================================
Install:
    pip install langsmith groq

Env vars:
    LANGSMITH_TRACING=true
    LANGSMITH_API_KEY=<your LangSmith key>
    GROQ_API_KEY=<your Groq key from console.groq.com>

On second+ runs: comment out the create_dataset / create_examples block.
"""

import os
from groq import Groq
from langsmith import Client

# ─────────────────────────────────────────────
# 1. Configure Groq
# ─────────────────────────────────────────────
groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])


def call_groq(
    prompt: str,
    model: str = "llama-3.3-70b-versatile",
    system: str = "",
) -> str:
    """Call a Groq-hosted model and return plain text."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = groq_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


# ─────────────────────────────────────────────
# 2. LangSmith client + dataset
# ─────────────────────────────────────────────
client_ls = Client()
DATASET_NAME = "QA ARISE AI Dataset (Groq)"

# ── Comment out after first run ──
dataset = client_ls.create_dataset(DATASET_NAME)
client_ls.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {
                "question": "What is JUSNL?"
            },
            "outputs": {
                "answer": "Jharkhand Urja Sancharan Nigam Limited (JUSNL) is the State Transmission Utility responsible for power transmission in Jharkhand."
            }
        },

        {
            "inputs": {
                "question": "Why has JUSNL filed this petition?"
            },
            "outputs": {
                "answer": "JUSNL filed this petition for provisional true-up of FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26, and tariff determination."
            }
        },

        {
            "inputs": {
                "question": "What regulations has JUSNL relied upon?"
            },
            "outputs": {
                "answer": "JUSNL relied upon the Electricity Act 2003 and JSERC Transmission Tariff Regulations 2020."
            }
        },

        {
            "inputs": {
                "question": "Why is JUSNL filing provisional true-up for FY 2023-24?"
            },
            "outputs": {
                "answer": "The statutory audit for FY 2023-24 was not completed, therefore JUSNL filed a provisional true-up based on unaudited accounts."
            }
        },

        {
            "inputs": {
                "question": "What petitions has JUSNL filed?"
            },
            "outputs": {
                "answer": "JUSNL filed Provisional True-Up for FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26 and Tariff Proposal for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What employee expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected total employee expenses of Rs. 107.54 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What A&G expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected A&G expenses of Rs. 25.45 Crore."
            }
        },

        {
            "inputs": {
                "question": "What depreciation expenses has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed depreciation expenses based on asset capitalization and applicable depreciation rates under JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What return on equity has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Return on Equity as per JSERC Transmission Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "What ARR has JUSNL projected for FY 2025-26?"
            },
            "outputs": {
                "answer": "JUSNL projected a Gross ARR of Rs. 475.40 Crore and a total revenue requirement of Rs. 1,418.88 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What transmission charges has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed monthly transmission charges to be recovered from beneficiaries based on the approved ARR."
            }
        },

        {
            "inputs": {
                "question": "What capital expenditure schemes has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed various transmission system strengthening and network expansion schemes."
            }
        },

        {
            "inputs": {
                "question": "What O&M expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected Operation and Maintenance expenses comprising employee cost, A&G expenses and R&M expenses."
            }
        },

        {
            "inputs": {
                "question": "What Repair and Maintenance expenses has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Repair and Maintenance expenses as part of O&M expenses under JSERC norms."
            }
        },

        {
            "inputs": {
                "question": "What interest on loan has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected interest on loan based on outstanding debt and applicable interest rates."
            }
        },

        {
            "inputs": {
                "question": "What interest on working capital has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Interest on Working Capital according to JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What is the regulatory framework applicable to JUSNL?"
            },
            "outputs": {
                "answer": "The regulatory framework consists of the Electricity Act 2003 and JSERC Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "How does JUSNL calculate employee expenses?"
            },
            "outputs": {
                "answer": "Employee expenses are projected using inflation escalation factors and estimated manpower additions."
            }
        },

        {
            "inputs": {
                "question": "How are terminal benefits treated in employee expenses?"
            },
            "outputs": {
                "answer": "Terminal benefits are separately accounted for and included in total employee expenses."
            }
        },

        {
            "inputs": {
                "question": "What are the major components of ARR?"
            },
            "outputs": {
                "answer": "ARR consists of O&M expenses, depreciation, interest on loan, return on equity and other approved costs."
            }
        }
    ]
,
)
print(f"✅ Dataset '{DATASET_NAME}' created with 5 examples.")
# ── End of first-run-only block ──


# ─────────────────────────────────────────────
# 3. Evaluators
# ─────────────────────────────────────────────
EVAL_SYSTEM = "You are an expert professor specialized in grading students' answers to questions."


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """LLM-as-a-judge: uses Groq/Llama to assess correctness."""
    prompt = f"""You are grading the following question:
{inputs['question']}

Here is the real answer:
{reference_outputs['answer']}

You are grading the following predicted answer:
{outputs['response']}

Respond with CORRECT or INCORRECT only — no explanation:
Grade:"""
    verdict = call_groq(prompt, system=EVAL_SYSTEM)
    return verdict.strip().upper() == "CORRECT"


def concision(outputs: dict, reference_outputs: dict) -> bool:
    """Response must be shorter than 2× the reference answer length."""
    return int(len(outputs["response"]) < 2 * len(reference_outputs["answer"]))


# ─────────────────────────────────────────────
# 4. App targets — three experiments
# ─────────────────────────────────────────────
DEFAULT_INSTRUCTIONS = (
    "Respond to the user's question in a short, concise manner (one short sentence)."
)
STRICT_INSTRUCTIONS = (
    "Respond to the user's question in a short, concise manner (one short sentence). "
    "Do NOT use more than ten words."
)


def my_app(
    question: str,
    model: str = "llama-3.3-70b-versatile",
    instructions: str = DEFAULT_INSTRUCTIONS,
) -> str:
    return call_groq(question, model=model, system=instructions)


# Experiment 1 — Llama 3.3 70B, default prompt
def target_llama70b(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="llama-3.3-70b-versatile")}


# Experiment 2 — Llama 3.1 8B (smaller/faster), default prompt
def target_llama8b(inputs: dict) -> dict:
    return {"response": my_app(inputs["question"], model="llama-3.1-8b-instant")}


# Experiment 3 — Llama 3.3 70B, strict prompt
def target_llama70b_strict(inputs: dict) -> dict:
    return {
        "response": my_app(
            inputs["question"],
            model="llama-3.3-70b-versatile",
            instructions=STRICT_INSTRUCTIONS,
        )
    }


# ─────────────────────────────────────────────
# 5. Run evaluations
# ─────────────────────────────────────────────
print("\n🚀 Experiment 1: llama-3.3-70b (default prompt) …")
r1 = client_ls.evaluate(
    target_llama70b,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="llama-70b-default",
)
print(f"   → {r1.experiment_name}")

print("\n🚀 Experiment 2: llama-3.1-8b (default prompt) …")
r2 = client_ls.evaluate(
    target_llama8b,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="llama-8b-default",
)
print(f"   → {r2.experiment_name}")

print("\n🚀 Experiment 3: llama-3.3-70b (strict prompt) …")
r3 = client_ls.evaluate(
    target_llama70b_strict,
    data=DATASET_NAME,
    evaluators=[concision, correctness],
    experiment_prefix="llama-70b-strict",
)
print(f"   → {r3.experiment_name}")

print("\n✅ Done. Open LangSmith → Datasets & Testing to compare results.")

## RAG Evaluation

In [ ]:
# 1) ensure project root is importable
import sys
import os
import os

from pathlib import Path

# Get the current folder path
current_folder = Path.cwd()
print("Current:", current_folder)

# Get the path of the folder one level up
parent_folder = current_folder.parent
print("Parent:", parent_folder)

# If you actually need to change your working directory to it:
import os
os.chdir(parent_folder)

# 2) imports
from sentence_transformers import SentenceTransformer
from src.config import CFG
from src.vector_db import collection
from src.generator import get_llm
from src.retriever import MongoHybridRetriever

# 3) build components
embedder = SentenceTransformer(CFG["embedding"]["model"])
llm = get_llm()

# 4) instantiate retriever
retriever = MongoHybridRetriever(
    embedder=embedder,
    llm=llm,
    collection=collection,
    use_mmr=True
)

# 5) call it (either .invoke or .get_relevant_documents)
docs = retriever.invoke("What employee expenses has JUSNL projected?")
# docs is a list of LangChain Document objects
for d in docs:
    print(d.metadata.get("_id"), d.page_content[:300])

In [ ]:
from langsmith import traceable

## add Decorator

# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.
       Use the following source documents to answer the user's questions.
       If you don't know the answer, just say that you don't know.
       Use three sentences maximum and keep the answer concise.

<context>
{docs_string}
</context>"""

    print("Number of docs:", len(docs))
    print("Characters:", len(docs_string))
    print("Approx tokens:", len(docs_string) // 4)
    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    )
    return {"answer": ai_msg.content, "documents": docs}



In [ ]:
rag_bot("What employee expenses has JUSNL projected?")

## Dataset for RAG

In [ ]:
from langsmith import Client
client_ls = Client()
DATASET_NAME = "RAG ARISE.AI Evaluation (Groq)"

# ── Comment out after first run ──
dataset = client_ls.create_dataset(DATASET_NAME)
client_ls.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {
                "question": "What is JUSNL?"
            },
            "outputs": {
                "answer": "Jharkhand Urja Sancharan Nigam Limited (JUSNL) is the State Transmission Utility responsible for power transmission in Jharkhand."
            }
        },

        {
            "inputs": {
                "question": "Why has JUSNL filed this petition?"
            },
            "outputs": {
                "answer": "JUSNL filed this petition for provisional true-up of FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26, and tariff determination."
            }
        },

        {
            "inputs": {
                "question": "What regulations has JUSNL relied upon?"
            },
            "outputs": {
                "answer": "JUSNL relied upon the Electricity Act 2003 and JSERC Transmission Tariff Regulations 2020."
            }
        },

        {
            "inputs": {
                "question": "Why is JUSNL filing provisional true-up for FY 2023-24?"
            },
            "outputs": {
                "answer": "The statutory audit for FY 2023-24 was not completed, therefore JUSNL filed a provisional true-up based on unaudited accounts."
            }
        },

        {
            "inputs": {
                "question": "What petitions has JUSNL filed?"
            },
            "outputs": {
                "answer": "JUSNL filed Provisional True-Up for FY 2023-24, APR for FY 2024-25, ARR for FY 2025-26 and Tariff Proposal for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What employee expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected total employee expenses of Rs. 107.54 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What A&G expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected A&G expenses of Rs. 25.45 Crore."
            }
        },

        {
            "inputs": {
                "question": "What depreciation expenses has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed depreciation expenses based on asset capitalization and applicable depreciation rates under JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What return on equity has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Return on Equity as per JSERC Transmission Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "What ARR has JUSNL projected for FY 2025-26?"
            },
            "outputs": {
                "answer": "JUSNL projected a Gross ARR of Rs. 475.40 Crore and a total revenue requirement of Rs. 1,418.88 Crore for FY 2025-26."
            }
        },

        {
            "inputs": {
                "question": "What transmission charges has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed monthly transmission charges to be recovered from beneficiaries based on the approved ARR."
            }
        },

        {
            "inputs": {
                "question": "What capital expenditure schemes has JUSNL proposed?"
            },
            "outputs": {
                "answer": "JUSNL proposed various transmission system strengthening and network expansion schemes."
            }
        },

        {
            "inputs": {
                "question": "What O&M expenses has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected Operation and Maintenance expenses comprising employee cost, A&G expenses and R&M expenses."
            }
        },

        {
            "inputs": {
                "question": "What Repair and Maintenance expenses has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Repair and Maintenance expenses as part of O&M expenses under JSERC norms."
            }
        },

        {
            "inputs": {
                "question": "What interest on loan has JUSNL projected?"
            },
            "outputs": {
                "answer": "JUSNL projected interest on loan based on outstanding debt and applicable interest rates."
            }
        },

        {
            "inputs": {
                "question": "What interest on working capital has JUSNL claimed?"
            },
            "outputs": {
                "answer": "JUSNL claimed Interest on Working Capital according to JSERC regulations."
            }
        },

        {
            "inputs": {
                "question": "What is the regulatory framework applicable to JUSNL?"
            },
            "outputs": {
                "answer": "The regulatory framework consists of the Electricity Act 2003 and JSERC Tariff Regulations."
            }
        },

        {
            "inputs": {
                "question": "How does JUSNL calculate employee expenses?"
            },
            "outputs": {
                "answer": "Employee expenses are projected using inflation escalation factors and estimated manpower additions."
            }
        },

        {
            "inputs": {
                "question": "How are terminal benefits treated in employee expenses?"
            },
            "outputs": {
                "answer": "Terminal benefits are separately accounted for and included in total employee expenses."
            }
        },

        {
            "inputs": {
                "question": "What are the major components of ARR?"
            },
            "outputs": {
                "answer": "ARR consists of O&M expenses, depreciation, interest on loan, return on equity and other approved costs."
            }
        }
    ]
,
)
print(f"✅ Dataset '{DATASET_NAME}' created with 5 examples.")
# ── End of first-run-only block ──

## Evaluators

### Correctness: Response vs reference answer

In [ ]:
from typing_extensions import Annotated, TypedDict

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grader_llm = llm.with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]

### Relevance: Response vs input

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool, ..., "Provide the score on whether the answer addresses the question"
    ]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = llm.with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs

In [ ]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[
        bool, ..., "Provide the score on if the answer hallucinates from the documents"
    ]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. You will be given FACTS and a STUDENT ANSWER. Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. (2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
grounded_llm = llm.with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["grounded"]

### Retrieval relevance: Retrieved docs vs input

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[
        bool,
        ...,
        "True if the retrieved documents are relevant to the question, False otherwise",
    ]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. You will be given a QUESTION and a set of FACTS provided by the student. Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = llm.with_structured_output(RetrievalRelevanceGrade, method="json_schema", strict=True)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"
    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer}
    ])
    return grade["relevant"]

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client_ls.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[correctness, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, groq_chatgptoss"},
)

# Explore results locally as a dataframe if you have pandas installed
# experiment_results.to_pandas()

In [ ]:
from src.evaluation_lang import dataset


dataset.creating_dataset()


In [ ]:
from src.evaluation_lang.evaluators import CorrectnessEvaluator
from src.evaluation_lang.llm_loader import llm_loader



judge_llm = llm_loader("gemini")

correctness_evaluator = CorrectnessEvaluator(
    llm=judge_llm,
    model_name="gemini"
)
correctness_evaluator


# result = correctness_evaluator(
#     inputs={
#         "question": "What ARR has JUSNL projected for FY 2025-26?"
#     },
#     outputs={
#         "answer": "The projected ARR is Rs. 1418.88 crore."
#     },
#     reference_outputs={
#         "answer": "The projected ARR for FY 2025-26 is Rs. 1418.88 crore."
#     }
# )

# print(result)

In [ ]:
# tests/test_gemini_api.py
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI


load_dotenv()

def test_gemini_basic():
    # Require GOOGLE_API_KEY or GEMINI_API in env or .env
    key = os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API")
    print(key)
    if not key:
        raise RuntimeError("Set GOOGLE_API_KEY or GEMINI_API in environment or .env")

    os.environ["GOOGLE_API_KEY"] = key  # ensure client finds it

    model = ChatGoogleGenerativeAI(
        model="models/gemini-2.5-flash",
        temperature=0
    )
    try:
        resp = model.invoke(
            [
                {"role": "system", "content": "You are a concise assistant."},
                {"role": "user", "content": "Say hello in one sentence."},
            ]
        )
    except Exception as e:
        raise RuntimeError(f"API call failed: {e}") from e

    assert hasattr(resp, "content")
    assert isinstance(resp.content, str) and resp.content.strip() != ""
    print("Gemini response:", resp.content)

In [ ]:
test_gemini_basic()

In [ ]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

for model in genai.list_models():
    if "generateContent" in model.supported_generation_methods:
        print(model.name)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    model="qwen/qwen3-32b"
)

In [ ]:
llm.invoke("hi").content

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ['OPEN_ROUTER_API_KEY']
)

models = client.models.list()

for m in models.data:
    # print(m.id)
    if ":free" in m.id:
        print(m.id)